# Physics-Informed Learning with a Galerkin Residual

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/poisson_galerkin.ipynb)

Instead of *solving* $Ku = F$, represent the solution by a small neural
network $u_\theta(x, y)$ and train it to drive the **discrete Galerkin
residual** to zero:

$$\min_\theta \; \|K u_\theta - F\|^2$$

TensorMesh assembles $K$ and $F$ once. Because `SparseMatrix.__matmul__` is
autograd-traced, the loss backpropagates straight into the network weights —
no linear solve in the loop, no hand-coded adjoint, and no collocation-point
sampling as in a classical PINN. The unique minimiser of the residual *is*
the FEM solution, so we can score the network against both the analytical
field and the direct solve.

Docs: [Physics-Informed Learning](https://docs.tensor-mesh.com/example_gallery/physics_informed.html) · Source: [`examples/physics_informed/poisson_galerkin.py`](https://github.com/camlab-ethz/TensorMesh/blob/main/examples/physics_informed/poisson_galerkin.py)

In [ ]:
# Install TensorMesh (skipped automatically if it is already available, e.g. a local dev setup).
# The apt line provides the OpenGL utility library that gmsh -- TensorMesh's mesh generator --
# needs at import time; it is a no-op where the library is already present.
import importlib.util
if importlib.util.find_spec("tensormesh") is None:
    !apt-get -qq install -y libglu1-mesa > /dev/null 2>&1 || true
    %pip install -q tensormesh-fem==0.2.0

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
import torch

from tensormesh import Condenser, LaplaceElementAssembler, MassElementAssembler, Mesh

torch.set_default_dtype(torch.float64)


class MLP(torch.nn.Module):
    """Coordinate network ``(x, y) -> u``: a plain tanh MLP."""

    def __init__(self, width=64, depth=3):
        super().__init__()
        layers = [torch.nn.Linear(2, width), torch.nn.Tanh()]
        for _ in range(depth - 1):
            layers += [torch.nn.Linear(width, width), torch.nn.Tanh()]
        layers += [torch.nn.Linear(width, 1)]
        self.net = torch.nn.Sequential(*layers)

    def forward(self, xy):
        return self.net(xy).squeeze(-1)

## Assemble once

The FEM operator is built a single time and then held fixed — it plays the
role that a differential operator plays in a classical PINN, but discretely
and exactly.

In [ ]:
H = 0.06            # mesh size
ADAM_ITERS = 8000
LBFGS_ITERS = 300

mesh = Mesh.gen_rectangle(chara_length=H)
x, y = mesh.points[:, 0], mesh.points[:, 1]
print(f"mesh: {mesh.n_points} nodes")

# Manufactured solution u = sin(pi x) sin(pi y)  =>  f = 2 pi^2 u.
u_exact = torch.sin(math.pi * x) * torch.sin(math.pi * y)
f_nodal = 2 * math.pi ** 2 * u_exact

# Assemble ONCE: stiffness K, consistent load F = M @ f, then condense.
K = LaplaceElementAssembler.from_mesh(mesh)().double()
M = MassElementAssembler.from_mesh(mesh)().double()
F = M @ f_nodal
cond = Condenser(mesh.boundary_mask)
K_, F_ = cond(K, F)
F_sq = (F_ ** 2).sum()

# The reference: the residual's exact minimiser is the ordinary FEM solution.
u_fem = cond.recover(K_.solve(F_))

interior = ~mesh.boundary_mask
coords = mesh.points[interior].clone()   # fixed network inputs

## Train

Adam warm-up followed by an LBFGS refinement — the usual recipe for this kind
of small deterministic objective.

In [ ]:
torch.manual_seed(0)
net = MLP().double()

hist = {"iter": [], "resid": [], "l2": []}


def metrics():
    with torch.no_grad():
        U = net(coords)
        rel_resid = (((K_ @ U - F_) ** 2).sum() / F_sq).item()
        rel_l2 = (torch.norm(cond.recover(U) - u_exact) / torch.norm(u_exact)).item()
    return rel_resid, rel_l2


def record(it):
    rr, l2 = metrics()
    hist["iter"].append(it); hist["resid"].append(rr); hist["l2"].append(l2)


# ---- Adam warm-up. The loss is the discrete Galerkin residual; SparseMatrix
# ---- @ is autograd-traced, so it backpropagates straight into the weights.
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
t0 = time.time()
for it in range(ADAM_ITERS):
    opt.zero_grad()
    loss = ((K_ @ net(coords) - F_) ** 2).sum() / F_sq
    loss.backward()
    opt.step()
    if it % 25 == 0:
        record(it)
    if it % 500 == 0:
        rr, l2 = metrics()
        print(f"  adam  {it:5d}  rel_resid={rr:.3e}  rel_L2={l2:.3e}")
print(f"adam: {ADAM_ITERS} iters in {time.time() - t0:.1f}s")

# ---- LBFGS refine.
opt2 = torch.optim.LBFGS(net.parameters(), lr=1.0, max_iter=LBFGS_ITERS,
                         history_size=50, line_search_fn="strong_wolfe")


def closure():
    opt2.zero_grad()
    loss = ((K_ @ net(coords) - F_) ** 2).sum() / F_sq
    loss.backward()
    return loss


opt2.step(closure)
record(ADAM_ITERS)

with torch.no_grad():
    u_nn = cond.recover(net(coords))
rel_fem = (torch.norm(u_nn - u_fem) / torch.norm(u_fem)).item()
print(f"final: rel_L2 vs exact = {hist['l2'][-1]:.3e},  vs FEM solve = {rel_fem:.3e}")

## Results

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4))
ax.semilogy(hist["iter"], hist["resid"], color="#c0392b", lw=2,
            label=r"relative Galerkin residual $\|K u_\theta - F\|^2 / \|F\|^2$")
ax.semilogy(hist["iter"], hist["l2"], color="#2980b9", lw=2, ls="--",
            label=r"relative $L^2$ error vs exact")
ax.set_xlabel("iteration  (Adam, then one LBFGS block)")
ax.set_title("Minimising the Galerkin residual")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

pts = mesh.points.numpy()
triang = mtri.Triangulation(pts[:, 0], pts[:, 1], mesh.cells["triangle"].numpy())
ue, un = u_exact.numpy(), u_nn.numpy()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
levels = np.linspace(0.0, 1.0, 21)
panels = [
    (ue, r"exact $u = \sin\pi x\,\sin\pi y$", "viridis", levels),
    (un, r"learned $u_\theta(x, y)$", "viridis", levels),
    (np.abs(un - ue), r"$|u_\theta - u|$", "Reds", 21),
]
for ax, (data, title, cmap, lv) in zip(axes, panels):
    cs = ax.tricontourf(triang, data, levels=lv, cmap=cmap)
    ax.set_aspect("equal")
    ax.set_title(title)
    fig.colorbar(cs, ax=ax, shrink=0.82)
plt.show()

## Where to next

- [Coefficient identification](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/coefficient_identification.ipynb) — autograd through the *solve* rather than the residual.
- [Dataset generation](https://docs.tensor-mesh.com/example_gallery/dataset.html) — batched FEM solves as training data for operator learning.